# Módulo 43: Temas avanzados de SQL

Autor: Juliho Castillo Colmenares

> Lección complementaria del ejercicio [m43-temas-avanzados-sql.ipynb](m43-temas-avanzados-sql.ipynb). No es una tarea entregada del programa.

Este módulo cierra la sección de SQL juntando todas las piezas en SQLite: diseñar tablas relacionadas con claves foráneas, cargarlas desde CSV con una función reutilizable y consultarlas con `JOIN`, `CASE` y subconsultas. Usamos `customers.csv` y `orders.csv`, dos archivos pequeños con clientes y sus pedidos.

1. Diseño de tablas relacionadas
2. Claves foráneas y `PRAGMA foreign_keys`
3. Carga genérica de CSV
4. `INNER JOIN` y `LEFT JOIN`
5. Semi-join y anti-join con `EXISTS`
6. Segmentar con `CASE` y `BETWEEN`
7. Subconsultas: escalares, en `FROM` y correlacionadas
8. Errores frecuentes
9. Ejercicio

## 1. Diseño de tablas relacionadas

Un cliente puede hacer muchos pedidos, y cada pedido pertenece a un solo cliente: es una relación **uno a muchos**. Se modela con dos tablas y una clave foránea en el lado "muchos".

| Tabla | Columna | Tipo | Significado |
| --- | --- | --- | --- |
| `Customers` | `customer_id` | `INTEGER` (PK) | Identificador del cliente |
| | `customer_name` | `TEXT` | Nombre |
| | `address`, `city`, `country` | `TEXT` | Domicilio |
| `Orders` | `order_id` | `INTEGER` (PK) | Identificador del pedido |
| | `customer_id` | `INTEGER` (FK) | Cliente que hizo el pedido |
| | `order_date` | `TEXT` | Fecha ISO `AAAA-MM-DD` |
| | `total_purchase` | `REAL` | Importe del pedido |

Declarar los tipos no es un detalle: como se verá en los errores frecuentes, una columna sin tipo guarda los números del CSV como texto.

In [1]:
import csv
import io
import sqlite3

import pandas as pd

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")   # se activa en cada conexión

conn.executescript("""
CREATE TABLE Customers (
    customer_id   INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    address       TEXT,
    city          TEXT,
    country       TEXT
);
CREATE TABLE Orders (
    order_id       INTEGER PRIMARY KEY,
    customer_id    INTEGER NOT NULL REFERENCES Customers(customer_id),
    order_date     TEXT,
    total_purchase REAL CHECK (total_purchase >= 0)
);
""")


def q(sql, params=None):
    """Ejecuta una consulta y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

## 2. Claves foráneas y `PRAGMA foreign_keys`

`REFERENCES Customers(customer_id)` declara que cada pedido debe apuntar a un cliente existente. Por compatibilidad, SQLite **no comprueba** las claves foráneas a menos que se active `PRAGMA foreign_keys = ON`, y la activación vale solo para esa conexión.

In [2]:
conn.execute("INSERT INTO Customers (customer_id, customer_name) VALUES (900, 'Cliente de prueba')")
conn.execute("INSERT INTO Orders VALUES (9000, 900, '2024-01-01', 10)")       # cliente existe: bien

try:
    conn.execute("INSERT INTO Orders VALUES (9001, 999, '2024-01-01', 10)")   # cliente 999 no existe
except sqlite3.IntegrityError as error:
    print("Rechazado:", error)

try:
    conn.execute("DELETE FROM Customers WHERE customer_id = 900")             # tiene un pedido
except sqlite3.IntegrityError as error:
    print("Rechazado:", error)

# Limpiar los datos de prueba (primero el pedido, luego el cliente)
conn.execute("DELETE FROM Orders WHERE order_id = 9000")
conn.execute("DELETE FROM Customers WHERE customer_id = 900")
conn.commit()

Rechazado: FOREIGN KEY constraint failed
Rechazado: FOREIGN KEY constraint failed


La clave foránea protege en los dos sentidos: no deja crear pedidos huérfanos ni borrar un cliente que todavía tiene pedidos.

## 3. Carga genérica de CSV

Cuando hay varias tablas que cargar, conviene una función reutilizable. Con `csv.DictReader`, los encabezados del archivo dan los nombres de columna, y `executemany` inserta todas las filas con parámetros con nombre.

Los nombres de tabla y columna no se pueden pasar como parámetros `?`; por eso la función comprueba que la tabla exista antes de armar la sentencia. Así nunca se ejecuta un nombre arbitrario.

In [3]:
def cargar_csv(conn, archivo, tabla):
    """Inserta las filas de un CSV (archivo abierto) en una tabla existente."""
    existe = conn.execute(
        "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?", (tabla,)).fetchone()
    if existe is None:
        raise ValueError(f"La tabla {tabla!r} no existe")

    lector = csv.DictReader(archivo)
    columnas = lector.fieldnames
    sql = (f"INSERT INTO {tabla} ({', '.join(columnas)}) "
           f"VALUES ({', '.join(':' + c for c in columnas)})")
    with conn:
        cur = conn.executemany(sql, lector)
    return cur.rowcount


with open("customers.csv", encoding="utf-8", newline="") as f:
    print("Clientes:", cargar_csv(conn, f, "Customers"))
with open("orders.csv", encoding="utf-8", newline="") as f:
    print("Pedidos:", cargar_csv(conn, f, "Orders"))

q("SELECT * FROM Customers LIMIT 4")

Clientes: 20
Pedidos: 40


,customer_id,customer_name,address,city,country
0,1,Juan Perez,Calle Falsa 123,Madrid,España
1,2,Maria Lopez,Avenida Siempre Viva 742,Bogotá,Colombia
2,3,Carlos Sanchez,Calle Larga 10,Lima,Perú
3,4,Ana Gomez,Calle Principal 22,México DF,México


Los archivos se abren con `encoding="utf-8"`. En Windows, `open` sin codificación usa la del sistema, y los acentos de `España` o `Bogotá` se leerían mal (sección 8).

Todos los clientes del archivo tienen al menos un pedido. Para que los `JOIN` muestren diferencias, agregamos dos clientes nuevos sin pedidos:

In [4]:
with conn:
    conn.executemany("INSERT INTO Customers VALUES (?, ?, ?, ?, ?)", [
        (21, "Lucía Fernández", "Calle Nueva 8", "Quito", "Ecuador"),
        (22, "Tomás Herrera", "Avenida Sur 300", "Asunción", "Paraguay"),
    ])

q("SELECT typeof(customer_id), typeof(total_purchase), COUNT(*) AS pedidos FROM Orders")

,typeof(customer_id),typeof(total_purchase),pedidos
0,integer,real,40


Gracias a los tipos declarados, los valores del CSV (que `csv` entrega como texto) se guardaron como números.

## 4. `INNER JOIN` y `LEFT JOIN`

- `INNER JOIN` responde "pedidos con los datos de su cliente": solo combinaciones existentes.
- `LEFT JOIN` con `Customers` a la izquierda responde "todos los clientes, con sus pedidos si los tienen".

Al resumir por cliente después de un `LEFT JOIN`, se cuenta `COUNT(o.order_id)` y no `COUNT(*)`: el primero ignora los `NULL` de los clientes sin pedidos.

In [5]:
q("""
SELECT c.customer_name, o.order_date, o.total_purchase
FROM Customers c
INNER JOIN Orders o ON o.customer_id = c.customer_id
ORDER BY o.order_date
LIMIT 5
""")

,customer_name,order_date,total_purchase
0,Juan Perez,2023-01-15,250.0
1,Maria Lopez,2023-02-10,100.0
2,Carlos Sanchez,2023-03-05,75.0
3,Ana Gomez,2023-03-20,500.0
4,Pedro Rodriguez,2023-04-11,700.0


In [6]:
q("""
SELECT c.customer_id, c.customer_name,
       COUNT(o.order_id)                 AS pedidos,
       COALESCE(SUM(o.total_purchase), 0) AS total
FROM Customers c
LEFT JOIN Orders o ON o.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total DESC, c.customer_id
""").tail(5)

,customer_id,customer_name,pedidos,total
17,18,Josefina Perez,2,350.0
18,19,Manuel Jimenez,2,350.0
19,1,Juan Perez,2,300.0
20,21,Lucía Fernández,0,0.0
21,22,Tomás Herrera,0,0.0


## 5. Semi-join y anti-join con `EXISTS`

Un **semi-join** devuelve las filas de una tabla que **tienen** pareja en otra, sin repetirlas ni traer columnas de la segunda. Un **anti-join** devuelve las que **no** la tienen. En SQL se escriben con subconsultas correlacionadas:

| Operación | Forma con `EXISTS` | Alternativa |
| --- | --- | --- |
| Semi-join | `WHERE EXISTS (SELECT 1 FROM B WHERE B.k = A.k)` | `WHERE A.k IN (SELECT k FROM B)` |
| Anti-join | `WHERE NOT EXISTS (SELECT 1 FROM B WHERE B.k = A.k)` | `LEFT JOIN B ... WHERE B.k IS NULL` |

In [7]:
# Clientes con al menos un pedido de más de 500 (semi-join)
display(q("""
SELECT c.customer_id, c.customer_name, c.country
FROM Customers c
WHERE EXISTS (SELECT 1 FROM Orders o
              WHERE o.customer_id = c.customer_id AND o.total_purchase > 500)
"""))

# Clientes sin ningún pedido (anti-join)
q("""
SELECT c.customer_id, c.customer_name
FROM Customers c
WHERE NOT EXISTS (SELECT 1 FROM Orders o WHERE o.customer_id = c.customer_id)
""")

,customer_id,customer_name,country
0,3,Carlos Sanchez,Perú
1,5,Pedro Rodriguez,Chile
2,9,Josefina Perez,Argentina
3,11,Marina Vargas,Perú
4,15,Josefina Perez,Argentina
5,17,Marina Vargas,Perú


,customer_id,customer_name
0,21,Lucía Fernández
1,22,Tomás Herrera


## 6. Segmentar con `CASE` y `BETWEEN`

`CASE` asigna una categoría a cada fila. Las condiciones se evalúan **en orden** y gana la primera verdadera, así que los rangos deben ser excluyentes o el orden debe resolver los bordes. `BETWEEN a AND b` incluye ambos extremos:

| Importe | Segmento |
| --- | --- |
| Menos de 100 | Bajo |
| De 100 a 400 (incluidos) | Medio |
| Más de 400 | Alto |

In [8]:
q("""
SELECT CASE
           WHEN total_purchase < 100 THEN 'Bajo'
           WHEN total_purchase BETWEEN 100 AND 400 THEN 'Medio'
           ELSE 'Alto'
       END                          AS segmento,
       COUNT(*)                     AS pedidos,
       MIN(total_purchase)          AS minimo,
       MAX(total_purchase)          AS maximo
FROM Orders
GROUP BY segmento
ORDER BY minimo
""")

,segmento,pedidos,minimo,maximo
0,Bajo,7,50.0,75.0
1,Medio,26,100.0,400.0
2,Alto,7,500.0,700.0


## 7. Subconsultas: escalares, en `FROM` y correlacionadas

| Tipo | Devuelve | Dónde se usa | Se evalúa |
| --- | --- | --- | --- |
| Escalar | Un solo valor | `SELECT`, `WHERE` | Una vez |
| En `FROM` (tabla derivada) | Una tabla | Como si fuera una tabla más, con alias | Una vez |
| Correlacionada | Depende de la fila externa | `SELECT`, `WHERE`, `EXISTS` | Conceptualmente, una vez por fila |

**Escalar:** pedidos por encima del promedio general, mostrando también ese promedio.

In [9]:
q("""
SELECT order_id, total_purchase,
       (SELECT ROUND(AVG(total_purchase), 2) FROM Orders) AS promedio_general
FROM Orders
WHERE total_purchase > (SELECT AVG(total_purchase) FROM Orders)
ORDER BY total_purchase DESC
LIMIT 5
""")

,order_id,total_purchase,promedio_general
0,105,700.0,280.63
1,111,600.0,280.63
2,117,600.0,280.63
3,123,600.0,280.63
4,129,600.0,280.63


**En `FROM`:** primero se calcula el total por país en una tabla derivada y después se consulta como cualquier tabla, aquí para quedarse con los países por encima de la media de países.

In [10]:
q("""
SELECT pais, total
FROM (
    SELECT c.country AS pais, SUM(o.total_purchase) AS total
    FROM Orders o
    JOIN Customers c ON c.customer_id = o.customer_id
    GROUP BY c.country
) AS por_pais
WHERE total > (SELECT AVG(total) FROM (
    SELECT SUM(o.total_purchase) AS total
    FROM Orders o JOIN Customers c ON c.customer_id = o.customer_id
    GROUP BY c.country))
ORDER BY total DESC
""")

,pais,total
0,Perú,4275.0
1,Argentina,2350.0
2,Venezuela,2150.0


**Correlacionada:** pedidos que superan el promedio **de su propio cliente**. La subconsulta usa `o.customer_id` de la fila externa, así que el umbral cambia en cada fila.

In [11]:
q("""
SELECT o.order_id, o.customer_id, o.total_purchase,
       (SELECT ROUND(AVG(o2.total_purchase), 1) FROM Orders o2
        WHERE o2.customer_id = o.customer_id) AS promedio_cliente
FROM Orders o
WHERE o.total_purchase > (SELECT AVG(o2.total_purchase) FROM Orders o2
                          WHERE o2.customer_id = o.customer_id)
ORDER BY o.customer_id
LIMIT 6
""")

,order_id,customer_id,total_purchase,promedio_cliente
0,101,1,250.0,150.0
1,122,2,400.0,250.0
2,123,3,600.0,337.5
3,104,4,500.0,325.0
4,105,5,700.0,500.0
5,126,6,200.0,175.0


## 8. Errores frecuentes

**Crear tablas sin tipos y comparar números.** Si una tabla se crea solo con nombres de columna (`CREATE TABLE t (a, b)`), los valores que llegan como texto desde `csv` se guardan como texto. En SQLite, un texto siempre es **mayor** que cualquier número, así que `total_purchase < 100` es falso para todos y el `CASE` manda todo al `ELSE`.

In [12]:
conn.execute("CREATE TABLE OrdersSinTipos (order_id, customer_id, order_date, total_purchase)")
with open("orders.csv", encoding="utf-8", newline="") as f:
    cargar_csv(conn, f, "OrdersSinTipos")

q("""
SELECT typeof(total_purchase) AS tipo,
       CASE WHEN total_purchase < 100 THEN 'Bajo'
            WHEN total_purchase BETWEEN 100 AND 400 THEN 'Medio'
            ELSE 'Alto' END AS segmento,
       COUNT(*) AS pedidos
FROM OrdersSinTipos
GROUP BY segmento
""")

,tipo,segmento,pedidos
0,text,Alto,40


Los 40 pedidos quedaron como "Alto". La solución es declarar los tipos al crear la tabla o, si no se puede, convertir en la consulta con `CAST(total_purchase AS REAL)`.

**Olvidar `PRAGMA foreign_keys = ON`.** Sin él, la cláusula `REFERENCES` es solo decorativa y se aceptan pedidos de clientes inexistentes:

In [13]:
otra = sqlite3.connect(":memory:")   # conexión nueva: claves foráneas desactivadas
otra.executescript("""
CREATE TABLE Customers (customer_id INTEGER PRIMARY KEY);
CREATE TABLE Orders (order_id INTEGER PRIMARY KEY,
                     customer_id INTEGER REFERENCES Customers(customer_id));
INSERT INTO Orders VALUES (1, 999);
""")
print("Pedidos huérfanos aceptados:", otra.execute("SELECT COUNT(*) FROM Orders").fetchone()[0])
otra.close()

Pedidos huérfanos aceptados: 1


**Abrir el CSV sin indicar la codificación.** El archivo está en UTF-8; leído como `cp1252` (la codificación habitual de Windows en español), cada letra acentuada se convierte en dos caracteres extraños:

In [14]:
with open("customers.csv", encoding="cp1252", newline="") as f:
    paises = [fila["country"] for fila in csv.DictReader(f)]
print(paises[:3])

['EspaÃ±a', 'Colombia', 'PerÃº']


**Contar con `COUNT(*)` después de un `LEFT JOIN`.** Un cliente sin pedidos aparece en una fila con `NULL`, y `COUNT(*)` la cuenta como un pedido. `COUNT(o.order_id)` da 0, que es lo correcto:

In [15]:
q("""
SELECT c.customer_name, COUNT(*) AS con_asterisco, COUNT(o.order_id) AS con_columna
FROM Customers c
LEFT JOIN Orders o ON o.customer_id = c.customer_id
WHERE c.customer_id IN (1, 21)
GROUP BY c.customer_id, c.customer_name
""")

,customer_name,con_asterisco,con_columna
0,Juan Perez,2,2
1,Lucía Fernández,1,0


## 9. Ejercicio

Una empresa quiere relacionar sus departamentos y empleados. Los datos llegan como dos CSV (aquí, en texto):

```python
departamentos_csv = """dept_id,dept_name
10,Finanzas
20,Tecnología
30,Ventas
40,Legal
"""
empleados_csv = """emp_id,emp_name,dept_id,salary
1,Ana,10,42000
2,Luis,20,51000
3,Marta,20,64000
4,Pedro,30,30500
5,Sofía,30,38000
6,Raúl,20,47000
7,Elena,10,58000
"""
```

1. Crea las tablas `Departamento` y `Empleado` con tipos, clave primaria y clave foránea, activa las claves foráneas y carga ambos textos con `cargar_csv` (usa `io.StringIO`).
2. Lista todos los departamentos con su número de empleados y su nómina total, incluidos los que no tienen empleados.
3. Encuentra los departamentos sin empleados con un anti-join.
4. Clasifica a cada empleado en `'Junior'` (menos de 40 000), `'Semi senior'` (de 40 000 a 55 000) o `'Senior'` (más de 55 000).
5. Lista los empleados que ganan más que el promedio de su departamento, con una subconsulta correlacionada.

### Solución

In [16]:
departamentos_csv = """dept_id,dept_name
10,Finanzas
20,Tecnología
30,Ventas
40,Legal
"""
empleados_csv = """emp_id,emp_name,dept_id,salary
1,Ana,10,42000
2,Luis,20,51000
3,Marta,20,64000
4,Pedro,30,30500
5,Sofía,30,38000
6,Raúl,20,47000
7,Elena,10,58000
"""

empresa = sqlite3.connect(":memory:")
empresa.execute("PRAGMA foreign_keys = ON")
empresa.executescript("""
CREATE TABLE Departamento (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL);
CREATE TABLE Empleado (
    emp_id   INTEGER PRIMARY KEY,
    emp_name TEXT NOT NULL,
    dept_id  INTEGER REFERENCES Departamento(dept_id),
    salary   REAL
);
""")
print("Departamentos:", cargar_csv(empresa, io.StringIO(departamentos_csv), "Departamento"))
print("Empleados:", cargar_csv(empresa, io.StringIO(empleados_csv), "Empleado"))

pd.read_sql_query("""
SELECT d.dept_name,
       COUNT(e.emp_id)          AS empleados,
       COALESCE(SUM(e.salary), 0) AS nomina
FROM Departamento d
LEFT JOIN Empleado e ON e.dept_id = d.dept_id
GROUP BY d.dept_id, d.dept_name
ORDER BY d.dept_id
""", empresa)

Departamentos: 4
Empleados: 7


,dept_name,empleados,nomina
0,Finanzas,2,100000.0
1,Tecnología,3,162000.0
2,Ventas,2,68500.0
3,Legal,0,0.0


In [17]:
display(pd.read_sql_query("""
SELECT d.dept_name
FROM Departamento d
WHERE NOT EXISTS (SELECT 1 FROM Empleado e WHERE e.dept_id = d.dept_id)
""", empresa))

pd.read_sql_query("""
SELECT emp_name, salary,
       CASE
           WHEN salary < 40000 THEN 'Junior'
           WHEN salary BETWEEN 40000 AND 55000 THEN 'Semi senior'
           ELSE 'Senior'
       END AS nivel
FROM Empleado
ORDER BY salary
""", empresa)

,dept_name
0,Legal


,emp_name,salary,nivel
0,Pedro,30500.0,Junior
1,Sofía,38000.0,Junior
2,Ana,42000.0,Semi senior
3,Raúl,47000.0,Semi senior
4,Luis,51000.0,Semi senior
5,Elena,58000.0,Senior
6,Marta,64000.0,Senior


In [18]:
pd.read_sql_query("""
SELECT e.emp_name, d.dept_name, e.salary
FROM Empleado e
JOIN Departamento d ON d.dept_id = e.dept_id
WHERE e.salary > (SELECT AVG(e2.salary) FROM Empleado e2 WHERE e2.dept_id = e.dept_id)
ORDER BY d.dept_id
""", empresa)

,emp_name,dept_name,salary
0,Elena,Finanzas,58000.0
1,Marta,Tecnología,64000.0
2,Sofía,Ventas,38000.0


In [19]:
empresa.close()
conn.close()

## Resumen

- Una relación uno a muchos se modela con una clave foránea en la tabla del lado "muchos"; en SQLite se hace cumplir solo con `PRAGMA foreign_keys = ON`.
- Declarar los tipos al crear las tablas evita que los números del CSV queden como texto y que las comparaciones fallen en silencio.
- `csv.DictReader` + `executemany` con parámetros con nombre permite una función de carga reutilizable; los CSV se abren con `encoding="utf-8"`.
- `EXISTS` y `NOT EXISTS` expresan semi-joins y anti-joins; `CASE` con `BETWEEN` segmenta, y el orden de los `WHEN` resuelve los bordes.
- Las subconsultas pueden devolver un valor (escalares), una tabla (en `FROM`) o depender de la fila externa (correlacionadas).

En el **Módulo 44** comienza la sección de Power BI con la preparación de datos.